## Functions to perform semantic preserving transformations of python code

In [1]:
#| default_exp semantic_preserving_transformations

### Variable renaming

In [ ]:
#| export
import libcst as cst
from redbaron import RedBaron
from baron.parser import ParsingError
import copy
import ast
import re
from libcst.metadata import ScopeProvider, MetadataWrapper, ParentNodeProvider
from transformers import RobertaTokenizer, RobertaForMaskedLM, pipeline
from libcst import (
    SimpleStatementLine,
    Assign,
    AssignTarget,
    BinaryOperation,
    Name,
    FlattenSentinel,
    TrailingWhitespace,
    Newline,
    SimpleWhitespace,
)


2025-02-11 20:51:08.677671: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739307068.695341 3726753 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739307068.700769 3726753 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-11 20:51:08.718682: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Renaming Transformations

In [ ]:
#| export
###############################################################################
# Transformer for rename_variable_1: First-letter renaming
###############################################################################
class FirstLetterVariableRenamer(cst.CSTTransformer):
    """
    A simple transformer that replaces variable names with their first letter.
    WARNING: This may lead to collisions if two identifiers share the same first letter.
    """
    METADATA_DEPENDENCIES = (ScopeProvider,)

    def leave_Name(
        self, original_node: cst.Name, updated_node: cst.Name
    ) -> cst.BaseExpression:
        # Only replace names longer than one character
        if len(updated_node.value) > 1:
            new_val = updated_node.value[0]
            return updated_node.with_changes(value=new_val)
        return updated_node

def rename_variable_1(code: str) -> str:
    """
    Replace variable names (and other identifiers) with their first letter.
    
    Args:
        code: A string containing Python source code.
    
    Returns:
        The transformed code as a string.
    """
    modified_code = code
    try :
        module = cst.parse_module(code)
        wrapper = MetadataWrapper(module)
        new_module = wrapper.visit(FirstLetterVariableRenamer())
        modified_code = new_module.code
    except Exception as e:
        print("An error occurred:", e)
    return modified_code

In [29]:
#| export

###############################################################################
# Transformer for rename_variable_2: CodeBERT-based renaming for variables only
###############################################################################

class CodeBERTVariableRenamer(cst.CSTTransformer):
    """
    Transformer that uses CodeBERT's fill-mask capability to generate new names,
    but only for variable names. It will update every occurrence (i.e. all references)
    of the same variable (as determined by its binding) while skipping:
      - function definitions,
      - class definitions,
      - function calls,
      - attribute accesses, and
      - import statements.
    
    For each identifier (that is not skipped) longer than one character, a dummy code context is
    constructed and CodeBERT is used to predict a suitable replacement. If the predicted token is
    not a valid identifier, it falls back to using the first letter.
    
    NOTE: This is an experimental approach and may result in unpredictable names.
    """
    # Request both scope and parent metadata.
    METADATA_DEPENDENCIES = (ScopeProvider, ParentNodeProvider)

    def __init__(self, model_name: str, cache_dir: str):
        # Load the tokenizer and model with the cache directory specified.
        model_tokenizer = RobertaTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
        pre_trained_model = RobertaForMaskedLM.from_pretrained(model_name, cache_dir=cache_dir, device_map='cpu')
        # Initialize the fill-mask pipeline.
        self.fill_mask = pipeline('fill-mask', model=pre_trained_model, tokenizer=model_tokenizer)
        # Cache substitutions keyed by a variable’s binding (or fallback to its name).
        self.substitutions = {}

    def leave_Name(self, original_node: cst.Name, updated_node: cst.Name) -> cst.BaseExpression:
        parent = self.get_metadata(ParentNodeProvider, original_node)

        # Skip renaming for import-related nodes.
        if isinstance(parent, (cst.Import, cst.ImportFrom, cst.ImportAlias)):
            return updated_node

        # Skip renaming for:
        # - The name of a function definition.
        # - The name of a class definition.
        # - The function being called.
        # - The attribute in an attribute access.
        if isinstance(parent, cst.FunctionDef) and parent.name == original_node:
            return updated_node
        if isinstance(parent, cst.ClassDef) and parent.name == original_node:
            return updated_node
        if isinstance(parent, cst.Call) and parent.func == original_node:
            return updated_node
        if isinstance(parent, cst.Attribute) and parent.attr == original_node:
            return updated_node

        # Retrieve the scope for this node and try to obtain its binding.
        try:
            scope = self.get_metadata(ScopeProvider, original_node)
            binding = scope.get_binding(original_node)
        except Exception:
            binding = None

        # Use the binding’s id as a key if available; otherwise, fallback to the name string.
        key = id(binding) if binding is not None else updated_node.value

        # Process only if the identifier is longer than one character.
        if len(updated_node.value) > 1:
            if key not in self.substitutions:
                original_name = updated_node.value
                # Create an enhanced dummy context.
                # This context provides extra semantic hints and only one <mask> token.
                dummy_context = (
                    f"def dummy({original_name}):\n"
                    f"    # This variable is used in a numerical computation. Suggest a clearer name.\n"
                    f"    result = {original_name} * 2\n"
                    f"    return <mask>"
                )
                try:
                    results = self.fill_mask(dummy_context)
                except Exception as e:
                    print(f"CodeBERT error for '{original_name}': {e}")
                    self.substitutions[key] = original_name[0]
                    return updated_node.with_changes(value=self.substitutions[key])
                
                if results:
                    # Check if the results are nested (i.e., if there are multiple mask tokens)
                    if isinstance(results[0], list):
                        # Use the first candidate from the first mask
                        new_name = results[0][0]["token_str"].strip()
                    else:
                        new_name = results[0]["token_str"].strip()
                    # Ensure the new name is a valid identifier.
                    if not new_name.isidentifier():
                        new_name = original_name[0]
                else:
                    new_name = original_name[0]
                self.substitutions[key] = new_name
            return updated_node.with_changes(value=self.substitutions[key])
        return updated_node

def rename_variable_2(code: str, model_name: str, cache_dir: str) -> str:
    """
    Replace variable names (and only variable names) using suggestions from CodeBERT,
    updating all references for a given variable binding.
    
    Args:
        code: A string containing Python source code.
        model_name: The name of the CodeBERT model.
        cache_dir: Directory path for caching the model.
    
    Returns:
        The transformed code as a string.
    """
    modified_code = code
    try:
        module = cst.parse_module(code)
        wrapper = cst.MetadataWrapper(module)
        new_module = wrapper.visit(CodeBERTVariableRenamer(model_name, cache_dir))
        modified_code =  new_module.code
    except Exception as e:
        print("An error occurred:", e)
    return modified_code


### Expression-level code Transformations

In [5]:
#| export
###############################################################################
# Transformer for switch_relation: Relational expression switching
###############################################################################

class RelationalExpressionSwitcher(cst.CSTTransformer):
    """
    Transformer that switches relational expressions.

    For example, transforms:
        a < b  -->  b > a

    Only comparisons with a single relational operator are transformed.
    """
    def leave_Comparison(self, original_node: cst.Comparison, updated_node: cst.Comparison) -> cst.BaseExpression:
        # Only handle comparisons with exactly one comparison target.
        if len(updated_node.comparisons) != 1:
            return updated_node

        comp = updated_node.comparisons[0]

        # Mapping of relational operators to their inverted counterparts.
        mapping = {
            cst.LessThan: cst.GreaterThan,
            cst.GreaterThan: cst.LessThan,
            cst.LessThanEqual: cst.GreaterThanEqual,
            cst.GreaterThanEqual: cst.LessThanEqual,
            # For equality and inequality the operator remains the same when operands are swapped.
            cst.Equal: cst.Equal,
            cst.NotEqual: cst.NotEqual,
            cst.Is: cst.Is,
            cst.IsNot: cst.IsNot,
        }

        op_type = type(comp.operator)
        if op_type in mapping:
            new_operator = mapping[op_type]()
        else:
            new_operator = comp.operator

        # Swap the operands:
        # - New left operand becomes the original comparator.
        # - The new comparison target is built using the original left operand.
        new_left = comp.comparator
        new_comparison = cst.ComparisonTarget(
            operator=new_operator,
            comparator=updated_node.left
        )

        return updated_node.with_changes(left=new_left, comparisons=[new_comparison])

def switch_relation(code: str) -> str:
    """
    Transform relational expressions by switching operands and inverting operators.

    For example, 'a < b' becomes 'b > a'.

    Args:
        code: Python source code as a string.

    Returns:
        The transformed code as a string.
    """
    modified_code = code
    try:
        module = cst.parse_module(code)
        wrapper = MetadataWrapper(module)
        new_module = wrapper.visit(RelationalExpressionSwitcher())
        modified_code = new_module.code
    except Exception as e:
        print("An error occurred:", e)
    return modified_code


In [6]:
#| export
###############################################################################
# Transformer for unary_2_add: Converting augmented assignments to explicit assignments
###############################################################################

class AugAssignTransformer(cst.CSTTransformer):
    """
    Transformer that converts augmented assignments (e.g., i += 1) into equivalent
    explicit assignments (e.g., i = i + 1).
    """
    def leave_AugAssign(
        self, original_node: cst.AugAssign, updated_node: cst.AugAssign
    ) -> cst.BaseSmallStatement:
        # Map the augmented assignment operator class names to their corresponding binary operator classes.
        operator_mapping = {
            "AddAssign": cst.Add,
            "SubtractAssign": cst.Subtract,
            "MultiplyAssign": cst.Multiply,
            "DivideAssign": cst.Divide,
            "ModuloAssign": cst.Modulo,
            "PowerAssign": cst.Power,
            "FloorDivideAssign": cst.FloorDivide,
            "BitAndAssign": cst.BitAnd,
            "BitOrAssign": cst.BitOr,
            "BitXorAssign": cst.BitXor,
            "LeftShiftAssign": cst.LeftShift,
            "RightShiftAssign": cst.RightShift,
        }
        # Use the operator's class name (e.g., "AddAssign") as the key.
        op_class_name = updated_node.operator.__class__.__name__
        binary_operator_class = operator_mapping.get(op_class_name, None)
        if binary_operator_class is None:
            # If the operator isn't recognized, leave the node unchanged.
            return updated_node

        # Construct a binary operation: <target> <binary_operator> <value>
        new_binary_op = cst.BinaryOperation(
            left=updated_node.target,
            operator=binary_operator_class(),
            right=updated_node.value,
        )
        # Construct an assignment: <target> = (<target> <binary_operator> <value>)
        new_assign = cst.Assign(
            targets=[cst.AssignTarget(target=updated_node.target)],
            value=new_binary_op,
        )
        return new_assign

def add_2_equal(code: str) -> str:
    """
    Transform augmented assignments (e.g., i += 1) into explicit assignments (e.g., i = i + 1).

    Args:
        code: Python source code as a string.

    Returns:
        The transformed code as a string.
    """
    modified_code = code
    try:
        # Parse the code into a CST.
        module = cst.parse_module(code)
        # Wrap the module to preserve formatting.
        wrapper = MetadataWrapper(module)
        # Visit the tree with our AugAssignTransformer.
        new_module = wrapper.visit(AugAssignTransformer())
        modified_code = new_module.code
    except Exception as e:
        print("An error occurred:", e)
    return modified_code

In [7]:
#| export
def infix_dividing(code: str) -> str:
    """
    Transform assignments of the form:
        x = L op1 R
    where R is itself a binary expression and its operator has higher precedence than op1,
    by extracting R into a temporary variable.
    
    For example, the code:
        x = a + b * c  # comment on assignment
    is transformed into:
        temp = b * c
        x = a + temp  # comment on assignment

    All comments and formatting are preserved.

    Args:
        code: The original Python source code.

    Returns:
        The transformed source code as a string.
    """
    modified_code = code
    try: 
        red = RedBaron(code)
    
        # Precedence mapping: operators as strings mapped to a numeric value.
        precedence_map = {
        "+": 1,
        "-": 1,
        "*": 2,
        "/": 2,
        "%": 2,
        "**": 3,
        }

        # Iterate over all assignment nodes.
        for assign in red.find_all("AssignmentNode"):
            # Check that the right-hand side is a binary operator node.
            if assign.value.type != "binary_operator":
                continue

            outer = assign.value
            tokens_outer = outer.dumps().split()
            if len(tokens_outer) < 3:
                continue  # not an expression like L op R

            # Assume the outer operator is the second token.
            op1 = tokens_outer[1]

            # Access the right-hand side (R). In RedBaron, for a binary operator node:
            #   - .first gives the left operand,
            #   - .second gives the right operand.
            R = outer.second

            # Only proceed if R is itself a binary operator node.
            if R.type != "binary_operator":
                continue

            tokens_R = R.dumps().split()
            if len(tokens_R) < 3:
                continue
            # Assume the operator in R is the second token.
            op_R = tokens_R[1]

            # Check precedence: extract if precedence(op_R) > precedence(op1)
            if precedence_map.get(op_R, 0) <= precedence_map.get(op1, 0):
                continue

            # Build a temporary assignment string: "temp = " + (dump of R)
            temp_assign_str = "temp = " + R.dumps()
            # Create a new node from the string.
            temp_assignment = RedBaron(temp_assign_str)[0]

            # Insert the temporary assignment before the current assignment.
            # Since assign.parent is a NodeList, we get the index of 'assign'
            # and then insert 'temp_assignment' at that index.
            parent_list = assign.parent
            index = parent_list.index(assign)
            parent_list.insert(index, temp_assignment)

            # Replace the original R (the right-hand side of the outer binary expression)
            # with the temporary variable "temp". Simply assigning "temp" creates a NameNode.
            outer.second = "temp"

        # Return the transformed code as a string.
        modified_code = red.dumps()
    except Exception as e:
        print("An error occurred:", e)
    return modified_code


In [8]:
#| export
class SwitchEqualExpTransformer(cst.CSTTransformer):
    """
    Transformer that switches the left and right expressions of a simple equality comparison.
    
    For a comparison of the form:
    
        a == b
    
    where the Comparison node has:
      - left: the left-hand side expression, and
      - comparisons: a list with a single ComparisonTarget whose operator is Equal and
        whose comparator is the right-hand side expression,
        
    this transformer returns a new Comparison node equivalent to:
    
        b == a
        
    The operator remains unchanged. Chained comparisons (with more than one operator)
    are not modified.
    """
    def leave_Comparison(
        self, original_node: cst.Comparison, updated_node: cst.Comparison
    ) -> cst.BaseExpression:
        # Process only simple (non-chained) comparisons.
        if len(updated_node.comparisons) != 1:
            return updated_node

        comp_target = updated_node.comparisons[0]
        # Check that the operator is an equality operator.
        if not isinstance(comp_target.operator, cst.Equal):
            return updated_node

        # Use copy.deepcopy to preserve formatting and any attached comments.
        new_left = copy.deepcopy(comp_target.comparator)
        new_comparator = copy.deepcopy(updated_node.left)

        # Create a new ComparisonTarget node with the same operator.
        new_comp_target = comp_target.with_changes(comparator=new_comparator)
        
        # Build a new Comparison node with swapped sides.
        return updated_node.with_changes(
            left=new_left,
            comparisons=[new_comp_target]
        )

def switch_equal_exp(code: str) -> str:
    """
    Switch the two expressions on both sides of a simple equality (==) comparison.
    
    For example, given the code:
    
        a == b
        
    the transformation produces:
    
        b == a
        
    Only simple (non‑chained) equality comparisons are modified; all comments,
    whitespace, and other code remain unchanged.
    
    Args:
        code: A string containing the original Python source code.
    
    Returns:
        A string containing the transformed Python source code.
    """
    modified_code = code
    try:
        # Parse the code into a CST.
        module = cst.parse_module(code)
        # Wrap the module; metadata is not required for this transformation.
        wrapper = MetadataWrapper(module)
        # Apply our transformer.
        new_module = wrapper.visit(SwitchEqualExpTransformer())
        # Return the modified code.
        modified_code = new_module.code
    except Exception as e:
        print("An error occurred:", e)
    return modified_code


### Statement-Level Transformations (Not Fully Functional)

In [15]:
#| hide
import libcst as cst
from libcst import FlattenSentinel

class For2WhileTransformer(cst.CSTTransformer):
    def leave_For(
        self, original_node: cst.For, updated_node: cst.For
    ) -> cst.FlattenSentinel[cst.BaseStatement]:
        # Only transform for-loops where the iterator is a call to range()
        if not (
            isinstance(updated_node.iter, cst.Call)
            and isinstance(updated_node.iter.func, cst.Name)
            and updated_node.iter.func.value == "range"
        ):
            return updated_node

        args = updated_node.iter.args
        # Determine the start, stop, and step expressions based on the number of arguments.
        if len(args) == 1:
            start_expr = cst.Integer("0")
            stop_expr = args[0].value
            step_expr = cst.Integer("1")
        elif len(args) == 2:
            start_expr = args[0].value
            stop_expr = args[1].value
            step_expr = cst.Integer("1")
        elif len(args) == 3:
            start_expr = args[0].value
            stop_expr = args[1].value
            step_expr = args[2].value
        else:
            # Unsupported range usage; leave the node unchanged.
            return updated_node

        # Create the initialization: <target> = <start_expr>
        # Propagate any leading comments (trivia) from the original for-loop.
        init_assign = cst.SimpleStatementLine(
            body=[
                cst.Assign(
                    targets=[cst.AssignTarget(target=updated_node.target)],
                    value=start_expr,
                )
            ],
            leading_lines=original_node.leading_lines  # reattach preceding comments
        )

        # Determine the appropriate comparison operator based on the step value.
        comparison_operator = cst.LessThan()
        if isinstance(step_expr, cst.Integer):
            try:
                step_val = int(step_expr.value)
                if step_val < 0:
                    comparison_operator = cst.GreaterThan()
            except ValueError:
                comparison_operator = cst.LessThan()
        else:
            # If the step isn't a constant, assume a positive step.
            comparison_operator = cst.LessThan()

        # Build the while loop condition: <target> < stop_expr (or > stop_expr if step is negative)
        condition = cst.Comparison(
            left=updated_node.target,
            comparisons=[
                cst.ComparisonTarget(
                    operator=comparison_operator, comparator=stop_expr
                )
            ],
        )

        # Create the increment statement: <target> += <step_expr>
        increment = cst.SimpleStatementLine(
            body=[
                cst.AugAssign(
                    target=updated_node.target,
                    operator=cst.AddAssign(),
                    value=step_expr,
                )
            ]
        )

        # Preserve the original body (including any inner comments) and append the increment.
        new_body = list(updated_node.body.body) + [increment]

        # Construct the while loop, preserving any 'else' clause.
        while_stmt = cst.While(
            test=condition,
            body=cst.IndentedBlock(body=new_body),
            orelse=updated_node.orelse,
        )

        # Return the initialization and while loop as replacement for the original for-loop.
        return FlattenSentinel([init_assign, while_stmt])

def for_2_while(source_code: str) -> str:
    """
    Transforms Python code containing for-loops over range() into equivalent while-loops.
    Only the for statements are changed, while preserving comments and all other code.
    
    Parameters:
      source_code: A string containing Python source code.
    
    Returns:
      A string with the transformed source code.
    """
    modified_code = source_code
    try:
        module = cst.parse_module(source_code)
        modified_module = module.visit(For2WhileTransformer())
        modified_code = modified_module.code
    except Exception as e:
        print("An error occurred:", e)
    return modified_code

In [24]:
#| hide
from redbaron import RedBaron
import re

def while_2_for(source_code: str) -> str:
    """
    Transforms simple counting while-loops into for-loops.

    Recognized pattern:
      - An initialization assignment for a variable (e.g. i = 0) occurs before a while-loop.
      - The while-loop’s condition is of the form "variable < limit" (or "variable > limit" for descending loops).
      - The last non-comment, non-empty statement in the loop body is an augmented assignment (e.g. i += 1 or i -= 1).
      
    Transformation:
      - Remove the initialization assignment.
      - Remove the update statement from the while body.
      - Replace the while-loop with a for-loop using range(start, limit, step).
      
    Note: Any code (including comments) that is not part of the transformed pattern is left intact.
    """
    modified_code = source_code
    try:
        red = RedBaron(source_code)
    
        # Iterate over all while-loop nodes
        for while_node in red.find_all("WhileNode"):
            condition_str = while_node.test.dumps().strip()
            # Expect condition of the form: "i < 5" or "j > 0"
            m = re.match(r"(\w+)\s*([<>]=?)\s*(.+)", condition_str)
            if not m:
                # Condition does not match the expected simple pattern
                continue
            cond_var, cond_op, limit_expr = m.groups()
            if cond_op not in ["<", ">"]:
                continue

            # Look for the initialization assignment immediately before the while-loop,
            # skipping any intervening comments or empty lines.
            parent = while_node.parent
            try:
                idx = parent.index(while_node)
            except Exception as e:
                print(f"Error finding index of while-loop: {e}")
                continue

            init_node = None
            # Scan backwards until we find an assignment node or hit a non-ignorable node.
            for j in range(idx - 1, -1, -1):
                candidate = parent[j]
                if candidate.type in ["comment", "endl"]:
                    continue
                if candidate.type == "assignment":
                    # Check that the target of the assignment matches the loop variable.
                    if candidate.target.dumps().strip() == cond_var:
                        init_node = candidate
                break  # Stop after the first non-comment/empty node
        
            if init_node is None:
                print(f"No initialization assignment found for variable '{cond_var}' before while-loop.")
                continue
        
            start_expr = init_node.value.dumps().strip()

            # Identify the update statement within the while body.
            if not while_node.value:
                print("While-loop body is empty; cannot find update statement.")
                continue

            last_stmt = None
            for node in reversed(while_node.value):
                if node.type in ["comment", "endl"]:
                    continue
                if node.type == "augassign":
                    last_stmt = node
                    break
            if not last_stmt:
                print("No valid augmented assignment update found in while-loop body.")
                continue

            update_target = last_stmt.target.dumps().strip()
            if update_target != cond_var:
                print(f"Update statement does not target the loop variable '{cond_var}'.")
                continue

            update_op = last_stmt.operator
            if cond_op == "<" and update_op != "+=":
                print("For an increasing loop, expected '+=' in update statement.")
                continue
            if cond_op == ">" and update_op != "-=":
                print("For a descending loop, expected '-=' in update statement.")
                continue

            step_expr = last_stmt.value.dumps().strip()
            # For descending loops, ensure the step is negative.
            if cond_op == ">":
                if not step_expr.startswith('-'):
                    step_expr = '-' + step_expr

            # Remove the update statement from the while body.
            while_node.value.remove(last_stmt)

            # Construct the new for-loop header using range(start, limit, step)
            new_for_header = f"for {cond_var} in range({start_expr}, {limit_expr}, {step_expr}):"
            # Build the for-loop code by combining the header with the (possibly multi-line) loop body.
            for_body = while_node.value.dumps()
            for_loop_code = new_for_header + "\n" + for_body
        
            # Create a new node from the for-loop code.
            new_for_node = RedBaron(for_loop_code)[0]
            # Replace the while-loop with the new for-loop.
            while_node.replace(new_for_node)
            # Remove the initialization assignment as it is now redundant.
            parent.remove(init_node)
    
        modified_code = red.dumps()
    except Exception as e:
        print("An error occurred:", e)
    return modified_code

### Real Data Example

In [17]:
#| hide
original_code = '''
# Module-level comment for demo purposes

def demo_function():
    # Starting the demo function
    print("Starting demo")
    
    # A simple loop counting from 0 to 3
    for i in range(4):  # loop index i
        # Print the index inside the loop
        print("Index:", i)  # inner loop comment
        
    # Another loop with a specified start and stop
    for j in range(2, 5):  # j goes from 2 to 4
        print("Value:", j)
    
    # Loop with a negative step: countdown from 10 to 6
    for k in range(10, 5, -1):  # countdown loop
        print("Countdown:", k)
        
    # This loop iterates over a list and should remain unchanged.
    for elem in ['x', 'y', 'z']:
        print("Element:", elem)  # list iteration comment
    
    # End of the demo function
    print("Demo completed")
    
demo_function()
'''
print("=== Original Complex Code ===")
print(original_code)

print("=== Transformed Code (For_2_While) ===")
print(for_2_while(original_code))

=== Original Complex Code ===

# Module-level comment for demo purposes

def demo_function():
    # Starting the demo function
    print("Starting demo")
    
    # A simple loop counting from 0 to 3
    for i in range(4):  # loop index i
        # Print the index inside the loop
        print("Index:", i)  # inner loop comment
        
    # Another loop with a specified start and stop
    for j in range(2, 5):  # j goes from 2 to 4
        print("Value:", j)
    
    # Loop with a negative step: countdown from 10 to 6
    for k in range(10, 5, -1):  # countdown loop
        print("Countdown:", k)
        
    # This loop iterates over a list and should remain unchanged.
    for elem in ['x', 'y', 'z']:
        print("Element:", elem)  # list iteration comment
    
    # End of the demo function
    print("Demo completed")
    
demo_function()

=== Transformed Code (For_2_While) ===

# Module-level comment for demo purposes

def demo_function():
    # Starting the demo function
    pr

In [25]:
#| hide
source_code = '''
# This is an example script

def count_up():
    # initialize counter
    i = 0  # start at 0
    # loop until i reaches 5
    while i < 5:
        print(f"Iteration {i}")  # print iteration
        # update counter
        i += 1

def count_down():
    # initialize counter for countdown
    j = 10
    # loop until j becomes 0
    while j > 0:
        print(f"Countdown: {j}")
        # update counter
        j -= 1

def no_transform():
    # This loop should not be transformed because the update is not the last statement
    k = 0
    while k < 3:
        print("Loop with extra update")
        k += 1
        k += 1  # extra update, so pattern not matched

# Some other code remains untouched
print("Script complete")
'''

print("=== Original Complex Code ===")
print(source_code)

print("=== Transformed Code (While_2_for) ===")
print(while_2_for(source_code))

=== Original Complex Code ===

# This is an example script

def count_up():
    # initialize counter
    i = 0  # start at 0
    # loop until i reaches 5
    while i < 5:
        print(f"Iteration {i}")  # print iteration
        # update counter
        i += 1

def count_down():
    # initialize counter for countdown
    j = 10
    # loop until j becomes 0
    while j > 0:
        print(f"Countdown: {j}")
        # update counter
        j -= 1

def no_transform():
    # This loop should not be transformed because the update is not the last statement
    k = 0
    while k < 3:
        print("Loop with extra update")
        k += 1
        k += 1  # extra update, so pattern not matched

# Some other code remains untouched
print("Script complete")

=== Transformed Code (While_2_for) ===
No valid augmented assignment update found in while-loop body.
No valid augmented assignment update found in while-loop body.
No valid augmented assignment update found in while-loop body.

# This is an 

In [31]:
#| hide
# Example of complex Python code as a multi-line string.
example_code = '''
import math

class DataProcessor:
    def __init__(self, items):
        self.items = items

    def filter_items(self):
        # List comprehension using a variable "item"
        filtered = [item for item in self.items if item % 2 == 0]
        return filtered

    def compute_sum(self, numbers):
        total = 0
        for num in numbers:
            total += num
        return total

def compute_complex():
    a_value = 10
    b_value = 20

    def inner_compute(x):
        # Using variables from the outer scope.
        temp = x * a_value
        return temp + b_value

    results = []
    for i in range(5):
        intermediate_result = inner_compute(i)
        results.append(intermediate_result)
    return results

def main():
    raw_data = [1, 2, 3, 4, 5, 6]
    processor = DataProcessor(raw_data)
    even_data = processor.filter_items()
    sum_even = processor.compute_sum(even_data)
    complex_results = compute_complex()
    
    print("Sum of even numbers:", sum_even)
    print("Complex computation results:", complex_results)

if __name__ == "__main__":
    main()
'''

# Example call to rename_variable_2 with the CodeBERT model name and a cache directory.
# (Make sure that 'microsoft/codebert-base' or your chosen model is accessible and that cache_dir exists.)
transformed_code = rename_variable_2(
    code=example_code, 
    model_name="microsoft/codebert-base", 
    cache_dir="/workspaces/CodeSmells/datax/hugging_face_cache"
)

print("=== Original Complex Code ===")
print(example_code)

print("=== Transformed Code ===")
print(transformed_code)


Some weights of RobertaForMaskedLM were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['lm_head.bias', 'lm_head.decoder.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


=== Original Complex Code ===

import math

class DataProcessor:
    def __init__(self, items):
        self.items = items

    def filter_items(self):
        # List comprehension using a variable "item"
        filtered = [item for item in self.items if item % 2 == 0]
        return filtered

    def compute_sum(self, numbers):
        total = 0
        for num in numbers:
            total += num
        return total

def compute_complex():
    a_value = 10
    b_value = 20

    def inner_compute(x):
        # Using variables from the outer scope.
        temp = x * a_value
        return temp + b_value

    results = []
    for i in range(5):
        intermediate_result = inner_compute(i)
        results.append(intermediate_result)
    return results

def main():
    raw_data = [1, 2, 3, 4, 5, 6]
    processor = DataProcessor(raw_data)
    even_data = processor.filter_items()
    sum_even = processor.compute_sum(even_data)
    complex_results = compute_complex()
    
    print("Sum